In [3]:
#VaR tests and vizualization
from scipy.stats import *
from scipy import stats 
from typing import Union, List, Literal, TypeAlias
import numpy as np
import numpy.typing as npt
import pandas as pd
from functools import wraps, partial
from arch import arch_model
import matplotlib.pyplot as plt

from Vares_simulations import *

from Vares import _terminal_returns, historical_var

Vares.ENABLE_TIMING = False

import pickle

#import the USD data 
with open('prices_usd.pkl', 'rb') as f: 
    prices_usd = pickle.load(f)

with open('original_returns_usd.pkl', 'rb') as f: 
    original_returns_usd = pickle.load(f)

with open('log_returns_usd.pkl', 'rb') as f: 
    log_returns_usd = pickle.load(f)    

original_returns_usd_scaled = original_returns_usd * 100
log_returns_usd_scaled = log_returns_usd * 100

from GARCH_VaR_delete_or_merge import fit_GARCH_VaR, normal_GARCH_simulation

import pickle

with open("norm_vars.pkl", "rb") as f:
    _norm_vars = pickle.load(f)

def turn_var_into_real_terms(var_array, prices):
    return [np.exp(log_delta) * price for log_delta, price in zip(var_array, prices)]

In [26]:
__FIGARCHSPECS__ = [[1, 2.0,1], [0, 2.0, 1], [1, 1.0, 1], [0, 1.0, 1], [1, 1.0, 0], [1 , 1.5, 1], [0, 2.0, 0]]
color = ['blue', 'red', 'yellow', 'green', 'purple', 'grey', 'brown']
from Vares_simulations import model_construction

    FIGARCH

In [28]:
figarch_models = {str(spec): model_construction('CM', 
                                                'FIGARCH', 
                                                'norm', 
                                                lags=None, 
                                                p=spec[0], 
                                                power=spec[1], 
                                                q=spec[2]) for spec in __FIGARCHSPECS__}

In [29]:
figarch_vars = {spec: fit_GARCH_VaR(log_returns_usd_scaled, figarch_models[spec]) for spec in figarch_models.keys()}

In [34]:
_figarch_vars = {spec: np.multiply(figarch_vars[spec], -0.01) for spec in figarch_vars.keys()}

In [ ]:
import plotly.graph_objects as go 

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=prices_usd, 
    mode='lines',
    line=dict(color='black', width=1.0), 
    name='Prices'
))

for color_i, model in enumerate(_figarch_vars.keys()): 
    fig.add_trace(go.Scatter(
        x=np.arange(len(prices_usd)),
        y=turn_var_into_real_terms(_figarch_vars[model], prices_usd),
        mode='lines',
        line=dict(color=color[color_i], width=0.5),
        name=model
    ))

fig.update_layout(
    title='FIGARCH', width=1400, height=700, hovermode='x unified', legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01)
)

fig.show()

In [50]:
#figarch difference 
figarch_diff = {spec: _figarch_vars[spec] / _norm_vars for spec in _figarch_vars.keys()}

import plotly.graph_objects as go 

fig = go.Figure()

for color_i, model in enumerate(figarch_diff.keys()): 
    fig.add_trace(go.Scatter(
        x=np.arange(365, 356+len(prices_usd)),
        y=figarch_diff[model][365:],
        mode='lines',
        line=dict(color=color[color_i], width=0.5),
        name=model
    ))

fig.update_layout(
    title='FIGARCH', width=1200, height=600, hovermode='x unified', legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01)
)

fig.show()

C:\Users\alexa\AppData\Local\Temp\ipykernel_4296\4294431850.py:2: RuntimeWarning:

invalid value encountered in divide



[1, 2.0, 1] and [0, 2.0, 1] - low diff => choose [1, 2.0, 1]
[1, 1.0, 1] and [0, 1.0, 1] - lof diff => choose [1, 1.0, 1]
Choose [1, 2.0, 1], [1, 1.0, 1], [1, 1.5, 1], [1, 1.0, 0]

In [51]:
__TARCHSPECS__ = [[1,0,1], [1, 1, 1], [2, 1, 2], [2, 0, 2], [2, 2, 2], [5, 0, 5], [5, 1, 5]]


tarch_models = {str(spec): model_construction('CM', 
                                                'TARCH', 
                                                'norm', 
                                                lags=None, 
                                                p=spec[0], 
                                                o=spec[1], 
                                                q=spec[2], power=1.0) for spec in __TARCHSPECS__}

In [54]:
tarch_vars = {spec: fit_GARCH_VaR(log_returns_usd_scaled, tarch_models[spec]) for spec in tarch_models.keys()}
_tarch_vars = {spec: np.multiply(tarch_vars[spec], -0.01) for spec in tarch_vars.keys()}

d:\Users\alexa\AppData\Local\Programs\Python\Python312\Lib\site-packages\arch\univariate\base.py:768: ConvergenceWarning:

The optimizer returned code 8. The message is:
Positive directional derivative for linesearch
See scipy.optimize.fmin_slsqp for code meaning.


d:\Users\alexa\AppData\Local\Programs\Python\Python312\Lib\site-packages\arch\univariate\base.py:768: ConvergenceWarning:

The optimizer returned code 8. The message is:
Positive directional derivative for linesearch
See scipy.optimize.fmin_slsqp for code meaning.


d:\Users\alexa\AppData\Local\Programs\Python\Python312\Lib\site-packages\arch\univariate\base.py:768: ConvergenceWarning:

The optimizer returned code 8. The message is:
Positive directional derivative for linesearch
See scipy.optimize.fmin_slsqp for code meaning.


d:\Users\alexa\AppData\Local\Programs\Python\Python312\Lib\site-packages\arch\univariate\base.py:768: ConvergenceWarning:

The optimizer returned code 8. The message is:
Positive directional derivati

In [56]:
import plotly.graph_objects as go 

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=prices_usd, 
    mode='lines',
    line=dict(color='black', width=1.0), 
    name='Prices'
))

for color_i, model in enumerate(_tarch_vars.keys()): 
    fig.add_trace(go.Scatter(
        x=np.arange(len(prices_usd)),
        y=turn_var_into_real_terms(_tarch_vars[model], prices_usd),
        mode='lines',
        line=dict(color=color[color_i], width=0.5),
        name=model
    ))

fig.update_layout(
    title='TARCH', width=1400, height=700, hovermode='x unified', legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01)
)

fig.show()

In [57]:
#figarch difference 
tarch_diff = {spec: _tarch_vars[spec] / _norm_vars for spec in _tarch_vars.keys()}

import plotly.graph_objects as go 

fig = go.Figure()

for color_i, model in enumerate(tarch_diff.keys()): 
    fig.add_trace(go.Scatter(
        x=np.arange(365, 356+len(prices_usd)),
        y=tarch_diff[model][365:],
        mode='lines',
        line=dict(color=color[color_i], width=0.5),
        name=model
    ))

fig.update_layout(
    title='FIGARCH', width=1200, height=600, hovermode='x unified', legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01)
)

fig.show()

C:\Users\alexa\AppData\Local\Temp\ipykernel_4296\834340337.py:2: RuntimeWarning:

invalid value encountered in divide



[1,0,1], [2,0,2], [5,0,5] are quite similar => only [1,0,1] remains
same for [1,1,1], [2,1,2], [2,2,2] and [5,1,5] => only [1,1,1]